# 📚 Bài Tập Thiết Kế Cơ Sở Dữ Liệu

**Dataset:** E-commerce Events (Tháng 10/2019 - Tháng 2/2020)

---

## 1. Các Loại Cơ Sở Dữ Liệu Sử Dụng

### 1.1 Tổng Quan

Dựa trên đề tài phân tích hành vi mua sắm E-commerce, nhóm sử dụng **3 loại cơ sở dữ liệu** phù hợp cho Big Data:

| # | Loại Database | Tên cụ thể | Mục đích sử dụng |
|---|---------------|------------|-------------------|
| 1 | **Relational DB** | PostgreSQL | Lưu trữ dữ liệu đã làm sạch, hỗ trợ SQL query |
| 2 | **Document Store** | MongoDB | Lưu trữ event data với schema linh hoạt |
| 3 | **Data Warehouse** | PostgreSQL (star schema) | Phân tích, reporting, BI |

---

### 1.2 Chi Tiết Lý Do Chọn Từng Loại

#### 🔷 PostgreSQL (Relational Database)

| Ưu điểm | Giải thích |
|----------|------------|
| **Miễn phí, open-source** | Phù hợp cho bài tập và project nhỏ |
| **Hỗ trợ SQL đầy đủ** | JOIN, GROUP BY, AGGREGATE dễ dàng |
| **ACID Compliance** | Đảm bảo tính nhất quán dữ liệu |
| **JSON Support** | Lưu trữ dữ liệu bán cấu trúc |
| **Kết nối tốt BI tools** | Power BI, Tableau, Grafana |

**Phù hợp cho:** Dữ liệu có cấu trúc rõ ràng (users, products, orders)

#### 🔷 MongoDB (Document Store / NoSQL)

| Ưu điểm | Giải thích |
|----------|------------|
| **Schema linh hoạt** | Mỗi event có thể có fields khác nhau |
| **Horizontal scaling** | Mở rộng khi data tăng |
| **JSON-like documents** | Native support cho event data |
| **Flexible queries** | Truy vấn phức tạp không cần schema cố định |

**Phù hợp cho:** Raw event data, session data, user behavior logs

#### 🔷 Data Warehouse (Star Schema)

| Ưu điểm | Giải thích |
|----------|------------|
| **Tối ưu cho phân tích** | Denormalized structure, fast queries |
| **Dimensional modeling** | Facts + Dimensions dễ hiểu |
| **BI friendly** | Aggregate tables cho dashboard |

**Phù hợp cho:** Báo cáo, dashboard, ML features

---

### 1.3 So Sánh Các Loại Database

| Tiêu chí | PostgreSQL | MongoDB | Data Warehouse |
|----------|------------|---------|-----------------|
| **Loại** | Relational | Document | Analytical |
| **Schema** | Fixed | Flexible | Star/Snowflake |
| **SQL Support** | Full | Limited | Full |
| **Scaling** | Vertical | Horizontal | Both |
| **Best for** | Transactions | Events | Analytics |
| **Phù hợp với bài này** | ✅ Users, Products | ✅ Raw Events | ✅ Reports |

---

## 2. Thiết Kế Cơ Sở Dữ Liệu

### 2.1 Database 1: PostgreSQL (Relational - Transactional)

#### Bảng `dim_users` (Dimension Table - Người dùng)

| Trường | Kiểu dữ liệu | Mô tả | Ví dụ |
|--------|---------------|--------|-------|
| user_id | BIGINT (PK) | ID người dùng | 463240011 |
| first_seen | TIMESTAMP | Lần đầu hoạt động | 2019-10-01 00:00:00 |
| last_seen | TIMESTAMP | Lần cuối hoạt động | 2020-02-29 23:59:59 |
| total_sessions | INT | Tổng số phiên | 15 |
| total_purchases | INT | Tổng số lần mua | 3 |
| total_spent | DECIMAL(10,2) | Tổng tiền đã chi | 156.75 |
| created_at | TIMESTAMP | Ngày tạo record | 2024-01-01 |

**Ví dụ 5 dòng dữ liệu:**

| user_id | first_seen | last_seen | total_sessions | total_purchases | total_spent | created_at |
|---------|------------|-----------|----------------|-----------------|-------------|------------|
| 463240011 | 2019-10-01 00:00:00 | 2020-02-15 14:30:00 | 12 | 5 | 245.50 | 2024-01-01 |
| 429681830 | 2019-10-01 00:00:07 | 2020-02-28 18:45:00 | 8 | 2 | 89.30 | 2024-01-01 |
| 377667011 | 2019-10-02 10:15:00 | 2020-01-20 09:00:00 | 3 | 1 | 15.00 | 2024-01-01 |
| 555446068 | 2019-10-05 16:20:00 | 2020-02-25 20:10:00 | 20 | 10 | 520.75 | 2024-01-01 |
| 546705258 | 2019-10-01 00:34:00 | 2020-02-29 22:00:00 | 6 | 3 | 180.25 | 2024-01-01 |

---

#### Bảng `dim_products` (Dimension Table - Sản phẩm)

| Trường | Kiểu dữ liệu | Mô tả | Ví dụ |
|--------|---------------|--------|-------|
| product_id | BIGINT (PK) | ID sản phẩm | 5773203 |
| category_id | BIGINT | ID danh mục | 1487580005134238553 |
| category_code | VARCHAR(100) | Mã danh mục | appliances.environment.vacuum |
| brand | VARCHAR(50) | Thương hiệu | runail |
| avg_price | DECIMAL(10,2) | Giá trung bình | 24.44 |
| min_price | DECIMAL(10,2) | Giá thấp nhất | 2.62 |
| max_price | DECIMAL(10,2) | Giá cao nhất | 150.00 |
| total_views | INT | Tổng lượt xem | 1500 |
| total_carts | INT | Tổng lượt thêm giỏ | 200 |
| total_purchases | INT | Tổng lượt mua | 50 |
| conversion_rate | DECIMAL(5,2) | Tỷ lệ chuyển đổi % | 3.33 |

**Ví dụ 5 dòng dữ liệu:**

| product_id | category_id | category_code | brand | avg_price | total_views | total_carts | total_purchases | conversion_rate |
|------------|-------------|---------------|-------|-----------|-------------|-------------|----------------|-----------------|
| 5773203 | 1487580005134238553 | cosmetics.nails.nail_polish | runail | 2.62 | 500 | 80 | 25 | 5.00 |
| 5881589 | 2151191071051219817 | cosmetics | lovely | 13.48 | 1200 | 150 | 45 | 3.75 |
| 5856191 | 1487580006350586771 | appliances.environment.vacuum | runail | 24.44 | 300 | 40 | 10 | 3.33 |
| 5739055 | 1487580008246412266 | cosmetics.nails | kapous | 4.75 | 800 | 100 | 30 | 3.75 |
| 5859414 | 1487580005671109489 | cosmetics.nails | masura | 2.37 | 2000 | 300 | 120 | 6.00 |

---

#### Bảng `dim_categories` (Dimension Table - Danh mục)

| Trường | Kiểu dữ liệu | Mô tả | Ví dụ |
|--------|---------------|--------|-------|
| category_id | BIGINT (PK) | ID danh mục | 1487580005134238553 |
| category_code | VARCHAR(100) | Mã danh mục | cosmetics.nails.nail_polish |
| category_level1 | VARCHAR(50) | Cấp 1 (parent) | cosmetics |
| category_level2 | VARCHAR(50) | Cấp 2 | nails |
| category_level3 | VARCHAR(50) | Cấp 3 | nail_polish |
| total_products | INT | Số sản phẩm | 500 |
| total_revenue | DECIMAL(12,2) | Tổng doanh thu | 150000.00 |

**Ví dụ 5 dòng dữ liệu:**

| category_id | category_code | category_level1 | category_level2 | category_level3 | total_products | total_revenue |
|-------------|---------------|-----------------|-----------------|-----------------|----------------|---------------|
| 1487580005134238553 | cosmetics.nails.nail_polish | cosmetics | nails | nail_polish | 1200 | 250000.00 |
| 2151191071051219817 | cosmetics | cosmetics | NULL | NULL | 3000 | 800000.00 |
| 1487580006350586771 | appliances.environment.vacuum | appliances | environment | vacuum | 150 | 45000.00 |
| 1487580008246412266 | cosmetics.nails | cosmetics | nails | NULL | 800 | 180000.00 |
| 1752742615205281895 | electronics | electronics | NULL | NULL | 2000 | 500000.00 |

---

#### Bảng `fact_events` (Fact Table - Sự kiện)

| Trường | Kiểu dữ liệu | Mô tả | Ví dụ |
|--------|---------------|--------|-------|
| event_id | BIGINT (PK) | ID sự kiện (auto) | 1 |
| event_time | TIMESTAMP | Thời gian sự kiện | 2019-10-01 00:00:00 |
| event_type | VARCHAR(20) | Loại sự kiện | view, cart, purchase |
| product_id | BIGINT (FK) | FK → dim_products | 5773203 |
| user_id | BIGINT (FK) | FK → dim_users | 463240011 |
| user_session | UUID | ID phiên làm việc | abc123-... |
| price | DECIMAL(10,2) | Giá tại thời điểm mua | 2.62 |

**Ví dụ 5 dòng dữ liệu:**

| event_id | event_time | event_type | product_id | user_id | user_session | price |
|----------|------------|------------|------------|---------|--------------|-------|
| 1 | 2019-10-01 00:00:00 | cart | 5773203 | 463240011 | 26dd6e6e-... | 2.62 |
| 2 | 2019-10-01 00:00:03 | cart | 5773353 | 463240011 | 26dd6e6e-... | 2.62 |
| 3 | 2019-10-01 00:00:07 | cart | 5881589 | 429681830 | 49e8d843-... | 13.48 |
| 4 | 2019-10-01 00:00:26 | view | 5875317 | 474232307 | 445f2b74-... | 1.59 |
| 5 | 2019-10-01 00:01:03 | purchase | 5677366 | 524009100 | 8bbff347-... | 7.30 |

---

### 2.2 Database 2: MongoDB (Document Store - Raw Events)

#### Collection: `raw_events`

```json
{
  "_id": ObjectId("..."),
  "event_time": "2019-10-01 00:00:00 UTC",
  "event_type": "cart",
  "product_id": 5773203,
  "category_id": 1487580005134238553,
  "category_code": "",
  "brand": "runail",
  "price": 2.62,
  "user_id": 463240011,
  "user_session": "26dd6e6e-4dac-4778-8d2c-92e149dab885"
}
```

**5 Ví dụ documents:**

```json
// Document 1
{
  "_id": ObjectId("5f8a1b2c3d4e5f6a7b8c9d0e"),
  "event_time": "2019-10-01 00:00:00 UTC",
  "event_type": "cart",
  "product_id": 5773203,
  "category_id": 1487580005134238553,
  "category_code": "",
  "brand": "runail",
  "price": 2.62,
  "user_id": 463240011,
  "user_session": "26dd6e6e-4dac-4778-8d2c-92e149dab885"
}

// Document 2
{
  "_id": ObjectId("5f8a1b2c3d4e5f6a7b8c9d0f"),
  "event_time": "2019-10-01 00:00:26 UTC",
  "event_type": "view",
  "product_id": 5875317,
  "category_id": 2029082628195353599,
  "category_code": "",
  "brand": "",
  "price": 1.59,
  "user_id": 474232307,
  "user_session": "445f2b74-5e4c-427e-b7fa-6e0a28b156fe"
}

// Document 3
{
  "_id": ObjectId("5f8a1b2c3d4e5f6a7b8c9d10"),
  "event_time": "2019-10-01 00:01:03 UTC",
  "event_type": "purchase",
  "product_id": 5677366,
  "category_id": 1752742615205281895,
  "category_code": "",
  "brand": "estel",
  "price": 7.30,
  "user_id": 524009100,
  "user_session": "8bbff347-0be1-470e-8860-d9a75db965b2"
}

// Document 4
{
  "_id": ObjectId("5f8a1b2c3d4e5f6a7b8c9d11"),
  "event_time": "2019-10-01 00:01:30 UTC",
  "event_type": "remove_from_cart",
  "product_id": 5834172,
  "category_id": 1487580013522845895,
  "category_code": "",
  "brand": "runail",
  "price": 0.95,
  "user_id": 429681830,
  "user_session": "49e8d843-adf3-428b-a2c3-fe8bc6a307c9"
}

// Document 5
{
  "_id": ObjectId("5f8a1b2c3d4e5f6a7b8c9d12"),
  "event_time": "2019-10-01 00:02:00 UTC",
  "event_type": "cart",
  "product_id": 5819638,
  "category_id": 1487580008162526182,
  "category_code": "electronics.tablet",
  "brand": "apple",
  "price": 599.99,
  "user_id": 546705258,
  "user_session": "3b5c65c0-bb1c-453b-b340-4ebf973a3136"
}
```

**Tại sao dùng MongoDB cho raw events:**
- Schema có thể thay đổi (missing fields, different types)
- Dễ dàng lưu trữ event data với các biến thể
- Flexible queries cho event analysis

---

### 2.3 Database 3: Data Warehouse (Star Schema - Analytics)

#### Fact Table: `fact_sales_daily`

| Trường | Kiểu dữ liệu | Mô tả |
|--------|---------------|--------|
| date_key | INT (PK, FK) | FK → dim_date |
| product_key | INT (FK) | FK → dim_product |
| brand_key | INT (FK) | FK → dim_brand |
| category_key | INT (FK) | FK → dim_category |
| total_views | INT | Tổng lượt xem |
| total_carts | INT | Tổng lượt thêm giỏ |
| total_purchases | INT | Tổng lượt mua |
| total_revenue | DECIMAL(12,2) | Tổng doanh thu |
| unique_users | INT | Số user unique |

**5 Ví dụ dòng dữ liệu:**

| date_key | product_key | brand_key | category_key | total_views | total_carts | total_purchases | total_revenue | unique_users |
|----------|-------------|-----------|--------------|-------------|-------------|----------------|---------------|--------------|
| 20191001 | 1001 | 101 | 201 | 5000 | 800 | 200 | 15000.00 | 450 |
| 20191001 | 1002 | 102 | 202 | 3000 | 500 | 100 | 8500.00 | 280 |
| 20191002 | 1001 | 101 | 201 | 5200 | 850 | 210 | 15800.00 | 470 |
| 20191002 | 1003 | 103 | 203 | 4500 | 600 | 150 | 12000.00 | 380 |
| 20191003 | 1002 | 102 | 202 | 3100 | 520 | 105 | 8900.00 | 295 |

---

#### Dimension Table: `dim_date`

| Trường | Kiểu dữ liệu | Mô tả |
|--------|---------------|--------|
| date_key | INT (PK) | 20191001 |
| full_date | DATE | 2019-10-01 |
| day_of_week | INT | 3 (Tuesday) |
| day_name | VARCHAR(10) | Tuesday |
| month | INT | 10 |
| month_name | VARCHAR(10) | October |
| quarter | INT | 4 |
| year | INT | 2019 |
| is_weekend | BOOLEAN | FALSE |

**5 Ví dụ:**

| date_key | full_date | day_of_week | day_name | month | month_name | quarter | year | is_weekend |
|----------|-----------|-------------|----------|-------|------------|---------|------|------------|
| 20191001 | 2019-10-01 | 3 | Tuesday | 10 | October | 4 | 2019 | FALSE |
| 20191002 | 2019-10-02 | 4 | Wednesday | 10 | October | 4 | 2019 | FALSE |
| 20191003 | 2019-10-03 | 5 | Thursday | 10 | October | 4 | 2019 | FALSE |
| 20191004 | 2019-10-04 | 6 | Friday | 10 | October | 4 | 2019 | FALSE |
| 20191005 | 2019-10-05 | 7 | Saturday | 10 | October | 4 | 2019 | TRUE |

---

#### Dimension Table: `dim_brand`

| Trường | Kiểu dữ liệu | Mô tả |
|--------|---------------|--------|
| brand_key | INT (PK) | 101 |
| brand_name | VARCHAR(50) | runail |
| brand_category | VARCHAR(50) | cosmetics |
| is_popular | BOOLEAN | TRUE |

**5 Ví dụ:**

| brand_key | brand_name | brand_category | is_popular |
|-----------|------------|----------------|------------|
| 101 | runail | cosmetics | TRUE |
| 102 | lovely | cosmetics | TRUE |
| 103 | masura | cosmetics | TRUE |
| 104 | kapous | cosmetics | FALSE |
| 105 | estel | cosmetics | FALSE |

---

### 2.4 Sơ Đồ Quan Hệ (ERD)

```
┌─────────────┐     ┌─────────────┐     ┌─────────────┐
│ dim_users   │     │ fact_events │     │ dim_products│
├─────────────┤     ├─────────────┤     ├─────────────┤
│ user_id (PK)│◄────│ user_id (FK)│     │product_id(PK)│
│ first_seen  │     │ event_id (PK)    │ brand       │
│ last_seen   │     │ event_time │────►│price        │
│ ...         │     │ event_type │     │ category_id │
└─────────────┘     │ product_id │     └──────┬──────┘
                    │ price      │            │
                    │ user_session     │            │
                    └─────────────┘     ┌────▼──────┐
                                      │dim_category│
                                      ├─────────────┤
                                      │category_id(PK)
                                      │category_code│
                                      │level1,2,3  │
                                      └─────────────┘
```

---

## 3. Số Lượng Điểm Dữ Liệu Cần Thiết Cho ML

### 3.1 Tổng Quan

**Câu trả lời:**

Tùy thuộc vào bài toán ML, số lượng data points cần thiết sẽ khác nhau:

| Bài toán ML | Data points tối thiểu | Giải thích |
|-------------|----------------------|------------|
| **Recommendation System** | **1,000 - 10,000** | Cần đủ user-item interactions |
| **Churn Prediction** | **5,000 - 50,000** | Cần đủ samples để phân biệt churn/non-churn |
| **Sales Forecasting** | **365+** (daily data) | Cần ít nhất 1 chu kỳ năm để học mùa vụ |
| **Customer Segmentation** | **1,000 - 5,000** | Clustering cần đủ density |
| **Price Optimization** | **10,000+** | Cần nhiều price points và outcomes |

---

### 3.2 Dataset Hiện Có

**Dataset E-commerce Events:**

| Thông tin | Giá trị |
|-----------|---------|
| **Tổng số events** | ~21 triệu rows (5 tháng) |
| **Unique users** | ~2 triệu users |
| **Unique products** | ~200,000 products |
| **Thời gian** | Oct 2019 - Feb 2020 (5 tháng) |

✅ **Đánh giá:** Dataset hiện có **RẤT ĐỦ** cho tất cả các bài toán ML

---

### 3.3 Chi Tiết Từng Bài Toán

#### 🎯 Bài toán 1: Recommendation System

| Tiêu chí | Giá trị |
|----------|---------|
| **Mục tiêu** | Gợi ý sản phẩm phù hợp cho user |
| **Thuật toán** | Collaborative Filtering, Matrix Factorization |
| **Data points cần** | 1,000 - 10,000 user-item interactions |
| **Dataset có** | ~21 triệu events ✅ |
| **Đủ không?** | ✅ **RẤT ĐỦ** |

---

#### 🎯 Bài toán 2: Churn Prediction

| Tiêu chí | Giá trị |
|----------|---------|
| **Mục tiêu** | Dự đoán user sẽ không quay lại |
| **Thuật toán** | Logistic Regression, Random Forest, XGBoost |
| **Data points cần** | 5,000 - 50,000 users (với labels) |
| **Dataset có** | ~2 triệu users ✅ |
| **Đủ không?** | ✅ **RẤT ĐỦ** |

---

#### 🎯 Bài toán 3: Sales Forecasting

| Tiêu chí | Giá trị |
|----------|---------|
| **Mục tiêu** | Dự đoán doanh số tương lai |
| **Thuật toán** | ARIMA, Prophet, LSTM |
| **Data points cần** | 365+ data points (daily) |
| **Dataset có** | ~150 ngày × nhiều products ✅ |
| **Đủ không?** | ✅ **ĐỦ** (có thể cần thêm seasonality) |

---

#### 🎯 Bài toán 4: Customer Segmentation

| Tiêu chí | Giá trị |
|----------|---------|
| **Mục tiêu** | Phân nhóm khách hàng theo hành vi |
| **Thuật toán** | K-Means, DBSCAN, Hierarchical Clustering |
| **Data points cần** | 1,000 - 5,000 customers |
| **Dataset có** | ~2 triệu users ✅ |
| **Đủ không?** | ✅ **RẤT ĐỦ** |

---

### 3.4 Kết Luận

```
┌─────────────────────────────────────────────────────────┐
│  ✅ Dataset E-commerce Events HOÀN TOÀN ĐỦ data cho ML  │
│                                                         │
│  • 21 triệu events → Rất nhiều cho training            │
│  • 2 triệu users → Đủ cho user-level ML                 │
│  • 5 tháng data → Đủ cho time-series analysis           │
│  • 200K products → Đủ cho product-level ML              │
└─────────────────────────────────────────────────────────┘
```

**Khuyến nghị:**
- Với dataset này, **không cần sampling** - có thể dùng toàn bộ
- Nên split train/test: **80/20** hoặc **70/30**
- Cross-validation: **5-fold** là đủ
- Monitor overfitting vì data rất nhiều

---

## 4. Tổng Kết

### Database sử dụng:
1. **PostgreSQL** - Relational DB cho users, products, categories, events
2. **MongoDB** - Document Store cho raw event data
3. **Data Warehouse (Star Schema)** - Cho analytics và BI

### Số điểm dữ liệu cho ML:
- **Recommendation:** 1,000 - 10,000 interactions
- **Churn Prediction:** 5,000 - 50,000 users
- **Sales Forecasting:** 365+ daily data points
- **Customer Segmentation:** 1,000 - 5,000 customers

### Dataset hiện có: ~21 triệu events ✅ (RẤT ĐỦ)